In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Matrix-State Recurrences

The selective models of that section decide, token by token, what to
keep and for how long. Much of what a language model must remember comes in
pairs, a name together with what it stands for. A program defines `rate =
0.05` near its top and uses `rate` two hundred lines later. A story
introduces a character as "the engineer" and refers to "the engineer" again
three pages on. To predict such a token, a model must store a *pair* and
later retrieve the second half of the pair when the first half reappears.
This is *associative recall*, and transformers do it with attention: the
current token's query is compared with the keys of all earlier tokens, and
the values of the best matches are returned. Can a recurrent state of fixed
size store pairs too, and how many?

This section makes the state a matrix that stores key–value pairs. A pair
is written by adding the outer product of its value and its key; it is read
by multiplying the matrix by a query. This is linear attention, the
variant of attention in that section that drops the
softmax, and, up to its decay, it is also what the selective state space
model of that section computes, with different names for the same
vectors.
We ask how many pairs such a memory can hold, how a decay keeps it from
filling up, and how to train it fast: a decayed matrix recurrence can be
computed as masked attention, as a scan, or in chunks that combine the two.
The section ends with a limit that the next section removes: every write
adds, so a memory cannot change what it has stored.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import random

## An Associative Memory

### Writing and Reading Pairs

The simplest memory for pairs is a sum: it adds each new pair to what it
holds and reads a value back with its key. A small example shows how this
works and where it fails. Start with two keys in $\mathbb{R}^2$,
$\mathbf{k}_1 = (1, 0)^\top$ and $\mathbf{k}_2 = (0, 1)^\top$, with the values $v_1 = 3$ and $v_2 = 5$.
Store both in one row vector, the sum of each value times its key,
$\mathbf{S} = v_1 \mathbf{k}_1^\top + v_2 \mathbf{k}_2^\top = (3, 5)$.
Multiplying by a key reads its value back: $\mathbf{S}\mathbf{k}_1 = 3$ and
$\mathbf{S}\mathbf{k}_2 = 5$. Now store a third pair, $\mathbf{k}_3 =
(1, 1)^\top / \sqrt{2}$ with $v_3 = 7$. The memory becomes $(3 + 7/\sqrt 2,
5 + 7/\sqrt 2) \approx (7.95, 9.95)$, and reading $\mathbf{k}_1$ now returns
7.95 instead of 3. Two dimensions hold two pairs exactly; a third key must
overlap with the others, and its value leaks into theirs.

With values $\mathbf{v}_t \in \mathbb{R}^{d_v}$ and keys $\mathbf{k}_t \in
\mathbb{R}^{d_k}$, the memory is a matrix $\mathbf{S}_t \in \mathbb{R}^{d_v
\times d_k}$, the transpose of the layout in
that section. The key width $d_k$ is the $d$ of
that section; the subscript keeps it apart from $d_v$, since both
widths appear in the shape of the memory. Each step writes one pair and the output
reads the memory with a query $\mathbf{q}_t \in \mathbb{R}^{d_k}$:

$$
\mathbf{S}_t = \mathbf{S}_{t-1} + \mathbf{v}_t \mathbf{k}_t^\top, \qquad
\mathbf{y}_t = \mathbf{S}_t \mathbf{q}_t.
$$

The outer product $\mathbf{v}_t \mathbf{k}_t^\top$ is a matrix of rank one
that maps the direction $\mathbf{k}_t$ to $\mathbf{v}_t$; the state is the
sum of these maps. The keys, values, and queries are linear projections of
the input, as in attention, so the network learns what to store under
which key and what to look up. A weight matrix that the network rewrites at
every step is called a *fast weight*; @Schlag.Irie.Schmidhuber.2021
trace linear attention back to this older idea.

### Linear Attention

The recurrence the equation does not look like attention, yet
it computes a form of it. Unrolling it from $\mathbf{S}_0 = \mathbf{0}$
gives

$$\mathbf{y}_t = \Big(\sum_{s \le t} \mathbf{v}_s \mathbf{k}_s^\top\Big)
\mathbf{q}_t = \sum_{s \le t} (\mathbf{k}_s^\top \mathbf{q}_t)\,
\mathbf{v}_s.$$

The right side is causal attention with the score $\mathbf{k}_s^\top
\mathbf{q}_t$ and no softmax: every earlier value, weighted by how well its
key matches the query. This is the *linear attention* of
the equation in that section, with the
identity as the feature map and without the normalizer (the models of this
chapter normalize the output of the layer instead): because the score is a
product of features rather than an exponential, the sum folds into a
fixed-size matrix, and the cost per token does not grow with the length of
the context [@Katharopoulos.Vyas.Pappas.ea.2020].
The left side is the recurrence. The same computation is a transformer
layer read one way and a recurrent network read the other.

### Interference

How many pairs can $\mathbf{S}$ hold? Store $n$ pairs, $\mathbf{S} =
\sum_{i=1}^n \mathbf{v}_i \mathbf{k}_i^\top$, with keys of unit length, and
read key $j$:

$$
\mathbf{S}\mathbf{k}_j = \mathbf{v}_j + \sum_{i \neq j}
(\mathbf{k}_i^\top \mathbf{k}_j)\, \mathbf{v}_i.
$$

The read returns the stored value plus every other value, weighted by the
overlap of its key with the target key. Only mutually orthogonal keys are
read back without error, and $\mathbb{R}^{d_k}$ holds at most $d_k$ of
them. For more pairs, or for keys that are not orthogonal, interference is
unavoidable. How large it is depends on the keys; for random keys we can
compute it.

**Proposition.** Let the keys $\mathbf{k}_1, \ldots, \mathbf{k}_n$ be
independent and uniformly distributed on the unit sphere of
$\mathbb{R}^{d_k}$, and let the values be unit vectors chosen independently
of the keys. Then the expected squared read error at every stored key is

$$
\mathbb{E}\,\big\|\mathbf{S}\mathbf{k}_j - \mathbf{v}_j\big\|^2 =
\frac{n - 1}{d_k}.
$$

**Proof.** The error is $\sum_{i \neq j} (\mathbf{k}_i^\top \mathbf{k}_j)\,
\mathbf{v}_i$. Its squared norm is a double sum over $i$ and $l$. For $i
\neq l$, the expected cross term vanishes: given $\mathbf{k}_j$, the
overlaps $\mathbf{k}_i^\top \mathbf{k}_j$ and $\mathbf{k}_l^\top
\mathbf{k}_j$ are independent with mean zero. What remains is $\sum_{i \neq
j} \mathbb{E}[(\mathbf{k}_i^\top \mathbf{k}_j)^2]$, using $\|\mathbf{v}_i\|
= 1$. A random unit vector has the same expected squared coordinate in every
direction, and these $d_k$ expectations sum to $\|\mathbf{k}_i\|^2 = 1$, so
each term is $1/d_k$. There are $n - 1$ terms. $\blacksquare$

The error grows in proportion to the number of stored pairs and falls in
proportion to the key dimension. Once $n$ approaches $d_k$, the
interference is as large as the signal, whose squared norm is one. Time
does not appear: a pair written long ago is read back as well as one
written a moment ago: memories are lost because other pairs interfere with
them, not because they are old.

We check the proposition by storing random pairs and reading them back.
The values come from a codebook of 256 random unit vectors, so we can also
ask whether the nearest codebook entry to the read is the stored one. A last
curve breaks the assumptions: every key shares a common direction, with
weight $\rho = 0.5$, so the keys are correlated. The computation needs only
NumPy.

In [ ]:
def matrix_memory(d_k, nums, rho=0, num_values=256, trials=20, seed=0):
    """Read error and recall of S = sum_i v_i k_i^T over random unit keys."""
    rng = np.random.default_rng(seed)
    unit = lambda X: X / np.linalg.norm(X, axis=-1, keepdims=True)
    codebook = unit(rng.standard_normal((num_values, d_k)))
    errors, recalls = [], []
    for n in nums:
        error = recall = 0
        for _ in range(trials):
            K = unit(rng.standard_normal((n, d_k)))  # Keys as rows
            K = unit(np.sqrt(1 - rho**2) * K + rho * unit(rng.standard_normal(d_k)))
            ids = rng.integers(0, num_values, n)
            S = codebook[ids].T @ K  # Sum of v_i k_i^T
            reads = K @ S.T  # Row j is S k_j
            error += ((reads - codebook[ids]) ** 2).sum(1).mean() / trials
            recall += ((reads @ codebook.T).argmax(1) == ids).mean() / trials
        errors.append(error)
        recalls.append(recall)
    return errors, recalls

In [ ]:
nums = [8, 16, 32, 64, 128, 256, 512]
results = {f'dₖ = {d_k}': matrix_memory(d_k, nums) for d_k in (32, 64, 128)}
results['dₖ = 64, ρ = 0.5'] = matrix_memory(64, nums, rho=0.5)
predictions = [[(n - 1) / d_k for n in nums] for d_k in (32, 64, 128)]
d2l.plot(nums, [err for err, _ in results.values()] + predictions,
         'stored pairs n', 'squared read error',
         legend=list(results) + ['$(n-1)/d_k$'], xscale='log', yscale='log',
         fmts=('-o', '-o', '-o', '-s', 'k:', 'k:', 'k:'), figsize=(5, 3.5))
print('recall at n =    ', ' '.join(f'{n:4d}' for n in nums))
for name, (_, recalls) in results.items():
    print(f'{name:17s}', ' '.join(f'{r:4.2f}' for r in recalls))

The measured errors lie on the predicted lines $(n-1)/d_k$ at all three
widths. The recall shows what this means for retrieval: with $d_k = 64$,
every pair is recovered up to $n = 64$, 96% at $n = 128$, 72% at $n = 256$,
and 37% at $n = 512$, and doubling $d_k$ doubles the number of pairs held
at a given recall. Correlated keys do worse. With a shared component of
weight $\rho = 0.5$, the error is about four times larger already at $n =
8$, it grows faster than in proportion to $n$, and only half of 64 pairs
are recovered. What a matrix memory can hold depends on how different its
keys are, not only on how many there are.

## Decay

### Forgetting Old Pairs

Linear attention never forgets. After $n$ writes all $n$ pairs share the
memory, and by the equation the read error grows with $n$,
however old most of the pairs are. A decay before each write shrinks the
old pairs,

$$
\mathbf{S}_t = a_t\, \mathbf{S}_{t-1} + \mathbf{v}_t \mathbf{k}_t^\top,
\qquad a_t \in (0, 1),
$$

so that a pair written $k$ steps ago has weight $a^k$ if the decay is
constant. The weights of all past pairs then sum to $1/(1 - a)$, and the
memory behaves as if it held about the last $1/(1-a)$ pairs: 10 for $a =
0.9$, 100 for $a = 0.99$. As in that section, a decay that depends on
the input lets the model choose: hold the memory through a paragraph and
clear it at a new document. Such a decay is the first of the two changes
that made linear attention competitive in practice
(that section); the second, keeping a few softmax
layers for recall, is the subject of that section.

The decay can be shaped in several ways, each a one-line change of
the equation. RetNet [@Sun.Dong.Huang.ea.2023] uses a fixed
scalar per head, so that different heads remember for different lengths.
Mamba-2 [@Dao.Gu.2024] uses an input-dependent scalar per head, as
above. Gated linear attention [@Yang.Wang.Shen.ea.2024] decays each key
coordinate at its own rate, $\mathbf{S}_t = \mathbf{S}_{t-1}
\textrm{diag}(\boldsymbol{\alpha}_t) + \mathbf{v}_t \mathbf{k}_t^\top$, so
that some directions of the key space are kept longer than others.

The selective state space model of that section is a matrix memory
of the same kind. Its state holds $N$ numbers for each of its $H$ channels,
an $H \times N$ matrix $\mathbf{S}_t$. Each step adds the outer product of the
channel inputs, scaled by the step sizes, $\boldsymbol{\Delta}_t \odot
\mathbf{x}_t$, with the vector $\mathbf{B}_t \in \mathbb{R}^N$, and the
output reads $\mathbf{S}_t \mathbf{C}_t$. In the terms of this section the
value is $\boldsymbol{\Delta}_t \odot \mathbf{x}_t$, the key is
$\mathbf{B}_t$, the query is $\mathbf{C}_t$, and $d_k = N$. Its decay is
elementwise, a different rate for every entry of $\mathbf{S}_t$. Mamba-2
replaced it by one scalar per head, which loses some flexibility and gains
the matrix products of the dual form below.

## The Dual Form

### Unrolling with Decay

Written as a recurrence, the decayed memory processes one token at a time.
Without decay, linear attention can also be computed as attention, over all
positions at once (that section); we ask whether this
survives the decay. With a scalar decay, unrolling the equation
from $\mathbf{S}_0 = \mathbf{0}$ gives each output as a weighted sum over the past, in which a
pair written at step $s$ is scaled by all decays met since:

$$\mathbf{y}_t = \sum_{s \le t} L_{ts}\, (\mathbf{k}_s^\top \mathbf{q}_t)\,
\mathbf{v}_s, \qquad L_{ts} = a_{s+1} a_{s+2} \cdots a_t.$$

Stack the queries, keys, values, and outputs of a sequence as the rows of
$\mathbf{Q}$, $\mathbf{K}$, $\mathbf{V}$, and $\mathbf{Y}$, and set $L_{ts}
= 0$ for $s > t$. Then all outputs together are

$$\mathbf{Y} = (\mathbf{L} \odot \mathbf{Q}\mathbf{K}^\top)\, \mathbf{V}.$$

This is masked attention. With every $a_t = 1$, $\mathbf{L}$ is the causal
mask of a transformer, ones on and below the diagonal, and
the equation is causal linear attention. With decays,
$\mathbf{L}$ is a *learned, decaying* mask: a decayed linear recurrence is
attention in which older tokens count less, by an amount the inputs choose.
@Dao.Gu.2024 call this correspondence *state space duality*. It
holds for linear attention, not for softmax attention, whose exponential
does not factor this way.

The two sides of the duality are two orders of computing the same
products. Forming $\mathbf{L} \odot \mathbf{Q}\mathbf{K}^\top$ first is
attention: all positions in parallel, at a cost and memory quadratic in
$T$. Summing from the left, one step at a time, is the recurrence: the
partial sums are the states $\mathbf{S}_t$, the cost is linear in $T$,
and the steps are sequential. We check that both give the same outputs,
starting with the recurrence.

In [ ]:
def linear_attention_recurrent(Q, K, V, a):
    """y_t = S_t q_t with S_t = a_t S_{t-1} + v_t k_t^T, step by step; a_t
    may also hold one decay per key coordinate, S_t = S_{t-1} diag(a_t) + ..."""
    def step(S, inputs):
        q, k, v, a_t = inputs
        S = a_t * S + jnp.outer(v, k)
        return S, S @ q
    S0 = jnp.zeros((V.shape[-1], K.shape[-1]))
    return jax.lax.scan(step, S0, (Q, K, V, a))[1]

The dual form needs the mask $\mathbf{L}$. Its entries are products of
decays, which we compute from the logarithms of the decays: $\log L_{ts}$ is
the sum of $\log a_r$ over $s < r \le t$, a difference of two cumulative
sums. Above the diagonal these differences are positive and their
exponentials can overflow, so we set them to $-\infty$ before
exponentiating, which turns them into exact zeros. We save the mask,
because the chunked algorithms below and in the next section use it too.

In [ ]:
def decay_mask(log_a):
    """L[t, s] = a_{s+1} * ... * a_t for s <= t, and 0 for s > t."""
    cum = log_a.cumsum(-1)
    log_L = cum[..., :, None] - cum[..., None, :]  # Sum of log a over (s, t]
    causal = jnp.tril(jnp.ones(log_L.shape[-2:], dtype=bool))
    return jnp.exp(jnp.where(causal, log_L, -jnp.inf))

def linear_attention_dual(Q, K, V, a):
    """The same outputs as Y = (L * Q K^T) V, all positions at once."""
    return (decay_mask(jnp.log(a)) * (Q @ K.T)) @ V

We compare the two on a random sequence of 512 steps with decays between
0.8 and 0.99.

In [ ]:
T, d_k, d_v = 512, 64, 64
key_q, key_k, key_v, key_a = jax.random.split(jax.random.key(0), 4)
Q, K = jax.random.normal(key_q, (T, d_k)), jax.random.normal(key_k, (T, d_k))
V = jax.random.normal(key_v, (T, d_v))
a = 0.8 + 0.19 * jax.random.uniform(key_a, (T,))
with jax.default_matmul_precision('highest'):  # Full float32, not TF32
    Y = linear_attention_recurrent(Q, K, V, a)
    Y_dual = linear_attention_dual(Q, K, V, a)
print(f'relative deviation: {jnp.abs(Y_dual - Y).max() / jnp.abs(Y).max():.1e}')

The two orders agree to rounding error. Neither is what we want for
training: the recurrence is sequential, and the dual form stores a $T
\times T$ matrix, 64 MB per head in 32-bit floats at $T = 4{,}096$.

### Chunked Computation

Chunking combines the two, as the equation of
that section did for linear attention without decay:
the quadratic form within each chunk, and a state carried from chunk to
chunk. The mask shows where the decays enter. Cut the sequence into chunks
of $m$ steps and partition the matrix $\mathbf{L} \odot
\mathbf{Q}\mathbf{K}^\top$ into $m \times m$ blocks, as in
the figure. The blocks on the diagonal are small masked
attention products, computed in parallel. The blocks below the diagonal are
never formed: everything a chunk needs from the entire past is the $d_v
\times d_k$ state at the chunk's start. With decays, the chunked computation
has three parts:

1. *Within chunks*: the dual form the equation on each
   chunk.
2. *Across chunks*: each chunk's writes, decayed to the chunk's end, form
   one matrix; the states at the chunk boundaries follow from a linear
   recurrence over the $T/m$ chunks, $\mathbf{S}^{(c)} = (\textrm{decay
   over chunk } c)\, \mathbf{S}^{(c-1)} + (\textrm{writes of chunk } c)$,
   which `linear_scan` of that section computes with matrix-valued
   states.
3. *Reading the past*: each position adds the read of the state at its
   chunk's start, decayed to its own position.

Almost all of the work is in matrix products, which GPUs do well, and the
memory holds $m \times m$ blocks and one state per chunk instead of a $T
\times T$ matrix. Mamba-2 and the models of the next section train this way.

![The mask $\mathbf{L}$ of the equation for random decays, partitioned into chunks. Diagonal blocks (blue) are small masked-attention products computed in parallel. Every block below the diagonal (orange) is accounted for by the state passed from chunk to chunk (arrows); the shading fades as the decays compound.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-modernrnn-semiseparable.svg)

A decay per key coordinate, $\mathbf{S}_t = \mathbf{S}_{t-1}
\textrm{diag}(\boldsymbol{\alpha}_t) + \mathbf{v}_t \mathbf{k}_t^\top$, as in
gated linear attention, changes only the first part. Let $c_{tj}$ be the
sum of $\log \alpha_{rj}$ from the chunk's first step to step $t$. The
decayed score of query $t$ and key $s \le t$ is then $\sum_j q_{tj}
k_{sj} \exp(c_{tj} - c_{sj})$: every key coordinate has a mask of its own,
and no single mask $\mathbf{L}$ multiplies $\mathbf{Q}\mathbf{K}^\top$.
`decay_mask` computes the $d_k$ masks at once, $d_k m^2$ numbers per chunk.

It is tempting to avoid them. Since $\exp(c_t - c_s) = \exp(c_t)
\exp(-c_s)$, scaling each query by $\exp(c_t)$ and each key by
$\exp(-c_s)$ turns the masked product of a chunk into a plain causal one.
In floating point this fold breaks. The factors $\exp(-c_s)$ grow as the
decays of a chunk compound and overflow 32-bit floats once those multiply
to less than $e^{-88}$, and learned gates get there: in gated linear
attention on the recall task of that section, the largest such
exponent grows from about 20 after 2,000 training steps to between 45 and
75 after 10,000, and two of three runs of 32,000 steps overflowed. One
overflow turns the gradients, and then every weight, into NaN, and the run
sits at chance from then on without raising any error. The function below never forms such
factors. Every exponential it computes is the decay over a stretch of
time, $\exp(c_t - c_s)$ with $s \le t$, $\exp(c_t)$, or $\exp(c_m - c_s)$,
and lies between 0 and 1, however strong the decays get.

The function takes a batch of sequences and the logarithms of the decays,
one per step or one per key coordinate. Its three parts follow the list
above; the decay from step $t$ to the end of its chunk is the exponential
of a difference of the cumulative log decays `cum`.

In [ ]:
def chunked_linear_attention(Q, K, V, log_a, chunk_size):
    """y_t = S_t q_t with S_t = S_{t-1} diag(a_t) + v_t k_t^T, in chunks.

    Q, K: (batch_size, T, d_k); V: (batch_size, T, d_v); log_a: (batch_size,
    T, d_k) for a decay per key coordinate, or (batch_size, T) for one per step
    """
    B, T, m = Q.shape[0], Q.shape[1], chunk_size
    if log_a.ndim == 2:  # The same decay for every key coordinate
        log_a = log_a[..., None]
    Q, K, V, log_a = (X.reshape(B, T // m, m, -1) for X in (Q, K, V, log_a))
    # 1. Within chunks: the dual form, with one mask per key coordinate
    L = decay_mask(log_a.mT)  # (B, T / m, 1 or d_k, m, m), entries <= 1
    if L.shape[2] == 1:  # One mask for all coordinates: a masked Q K^T
        Y = (L[:, :, 0] * (Q @ K.mT)) @ V
    else:
        Y = (L * Q.mT[..., None] * K.mT[..., None, :]).sum(2) @ V
    # 2. Across chunks: the writes of each chunk, decayed to its end, and
    #    a scan over chunks for the state at every chunk's end
    cum = log_a.cumsum(2)  # Log decay from the chunk's start to each step
    writes = V.mT @ (K * jnp.exp(cum[:, :, -1:] - cum))
    S = d2l.linear_scan(jnp.exp(cum[:, :, -1:]), writes)
    S = jnp.concatenate([jnp.zeros_like(S[:, :1]), S[:, :-1]], 1)  # At start
    # 3. Every step reads the state at its chunk's start, decayed to it
    return (Y + (Q * jnp.exp(cum)) @ S.mT).reshape(B, T, -1)

The chunked outputs match the recurrence for every chunk size, with one
decay per step and with one per key coordinate.

In [ ]:
a_key = 0.8 + 0.19 * jax.random.uniform(  # One decay per key coordinate
    jax.random.key(2), (T, d_k))
with jax.default_matmul_precision('highest'):  # Full float32, not TF32
    decays = {'per step': (a, Y),
              'per key': (a_key, linear_attention_recurrent(Q, K, V, a_key))}
    for m in [16, 64, 256]:
        for name, (decay, Y_ref) in decays.items():
            Y_chunked = chunked_linear_attention(
                Q[None], K[None], V[None], jnp.log(decay)[None], m)[0]
            print(f'chunk size {m:3d}, decay {name}: relative deviation '
                  f'{jnp.abs(Y_chunked - Y_ref).max() / jnp.abs(Y_ref).max():.1e}')

How fast are the three orders? We time them on one sequence of 4,096 steps
on a GPU.

In [ ]:
T = 4096
key_q, key_k, key_v, key_a = jax.random.split(jax.random.key(1), 4)
Q, K = jax.random.normal(key_q, (T, d_k)), jax.random.normal(key_k, (T, d_k))
V = jax.random.normal(key_v, (T, d_v))
a = 0.8 + 0.19 * jax.random.uniform(key_a, (T,))
forms = {'recurrence': jax.jit(linear_attention_recurrent),
         'dual': jax.jit(linear_attention_dual)}
for m in [16, 64, 256]:
    forms[f'chunks of {m}'] = jax.jit(lambda Q, K, V, a, m=m:
        chunked_linear_attention(Q[None], K[None], V[None], jnp.log(a)[None], m))
for name, f in forms.items():
    bench = d2l.Benchmark(lambda: f(Q, K, V, a), warmup=1, repeats=5)
    print(f'{name:14s} {1e3 * bench.time:8.2f} ms')

The recurrence is the slowest by two orders of magnitude, because its 4,096
steps run one after another. At this length the dual form and the chunked
form take similar times, about a millisecond or less. They differ in
memory: the dual form stores the $T \times T$ matrix of weights, 64 MB per
head in 32-bit floats, which grows with the square of the length, while
chunks of 64 store $T \times 64$ weights and one $64 \times 64$ state per
chunk, about 2 MB, which grows in proportion to the length.

## A Gated Linear Attention Layer

### The Layer

A layer built on the decayed matrix memory computes, for every head, a
query, a key, a value, and decays from the input, runs the chunked
recurrence, and projects the outputs of all heads back to the model width.
We call it *gated linear attention*, after its decay gate, and build it as
@Yang.Wang.Shen.ea.2024 and the layers that followed it do. Four
parts surround the memory; the experiments below and in
that section measure what they contribute:

- *A short convolution.* The projected queries, keys, and values pass
  through a causal convolution of width 4, one filter per channel, and a
  SiLU. Each of them can then draw on the few tokens before it, so that a
  pair spread over neighboring tokens, a name followed by its value, can
  be written as one key and one value.
- *Unit queries and keys,* the setting of the equation.
- *A decay per key coordinate*, $\boldsymbol{\alpha}_t =
  \sigma(\mathbf{W}_2\mathbf{W}_1\mathbf{x}_t + \mathbf{b})^{1/16}$, computed
  through a projection of rank 16. The exponent $1/16$ keeps the decays
  close to one while the gate moves, and the biases start the coordinates
  with memories of 4 to 256 steps, spread log-uniformly, as the timescales
  of that section were.
- *An output norm and gate.* The output of each head is normalized to unit
  root mean square, which keeps its scale independent of how many pairs the
  memory holds, and multiplied by a gate $\textrm{SiLU}(\mathbf{W}_g
  \mathbf{x}_t)$ before the output projection.

In [ ]:
class GatedLinearAttention(nnx.Module):
    """Multi-head linear attention with a gated decay per key coordinate."""
    gate_scale = 16  # log a = log sigmoid(z) / 16 keeps decays near one

    def __init__(self, num_hiddens, num_heads, chunk_size=16, conv_width=4,
                 timescales=(4, 256), *, rngs):
        d = num_hiddens
        self.num_heads, self.chunk_size = num_heads, chunk_size
        self.W_qkv = nnx.Linear(d, 3 * d, use_bias=False, rngs=rngs)
        self.conv = nnx.Conv(3 * d, 3 * d, (conv_width,),
                             feature_group_count=3 * d,  # One filter each
                             padding=((conv_width - 1, 0),), rngs=rngs)  # Causal
        self.W_a = self.decay_gate(d, rngs)
        self.W_g = nnx.Linear(d, d, rngs=rngs)  # Output gate
        self.W_o = nnx.Linear(d, d, use_bias=False, rngs=rngs)
        self.init_decays(*timescales)  # Memories of 4 to 256 steps by default

    def decay_gate(self, d, rngs, rank=16):
        """Low-rank projection to one gate per key coordinate."""
        return nnx.Sequential(nnx.Linear(d, rank, use_bias=False, rngs=rngs),
                              nnx.Linear(rank, d, rngs=rngs))

    def init_decays(self, tau_min, tau_max):
        """Start the gates at decays exp(-1/tau), tau log-uniform in range."""
        b = self.W_a.layers[-1].bias
        tau = jnp.logspace(math.log10(tau_min), math.log10(tau_max), b.shape[0])
        b[...] = -jnp.log(jnp.expm1(self.gate_scale / tau))

The recurrence sits in a method of its own, `memory`, and the decays in
`decays`, so that the next section can replace them. Both see the heads
side by side: `heads_to_batch` moves the heads of a minibatch into its
batch axis, so that the chunked computation treats every head as a
sequence of its own, and `batch_to_heads` moves them back.

In [ ]:
def heads_to_batch(Z, num_heads):
    """(batch_size, T, num_heads * e) -> (batch_size * num_heads, T, e)."""
    B, T, _ = Z.shape
    return Z.reshape(B, T, num_heads, -1).swapaxes(1, 2).reshape(
        B * num_heads, T, -1)

def batch_to_heads(Y, batch_size):
    """(batch_size * num_heads, T, e) -> (batch_size, T, num_heads * e)."""
    _, T, e = Y.shape
    return Y.reshape(batch_size, -1, T, e).swapaxes(1, 2).reshape(
        batch_size, T, -1)

@d2l.add_to_class(GatedLinearAttention)
def decays(self, X):
    """Log decays, (batch_size * num_heads, T, d_k)."""
    return heads_to_batch(jax.nn.log_sigmoid(self.W_a(X)) / self.gate_scale,
                          self.num_heads)

@d2l.add_to_class(GatedLinearAttention)
def memory(self, X, Q, K, V, log_a):
    """Outputs of the decayed matrix memory of every head. X is unused here;
    the delta rule of the next section reads its write strengths from it."""
    return chunked_linear_attention(Q, K, V, log_a, self.chunk_size)

The forward pass runs the projections through the convolution and the
SiLU, splits them into the queries, keys, and values of the heads, scales
queries and keys to unit length, runs the memory, normalizes and gates the
output of each head, and projects the concatenated heads back to the model
width.

In [ ]:
@d2l.add_to_class(GatedLinearAttention)
def __call__(self, X):  # X: (batch_size, num_steps, num_hiddens)
    QKV = jax.nn.silu(self.conv(self.W_qkv(X)))
    Q, K, V = (heads_to_batch(Z, self.num_heads)
               for Z in jnp.split(QKV, 3, -1))
    Q, K = (Z / jnp.maximum(jnp.linalg.norm(Z, axis=-1, keepdims=True), 1e-12)
            for Z in (Q, K))  # Unit length
    Y = self.memory(X, Q, K, V, self.decays(X))  # (B * num_heads, T, d / h)
    Y = Y / jnp.sqrt((Y ** 2).mean(-1, keepdims=True) + 1e-6)  # Unit RMS
    return self.W_o(batch_to_heads(Y, X.shape[0]) * jax.nn.silu(self.W_g(X)))

### Language Modeling

We train the layer with four heads of dimension 32 as the mixer of
`d2l.SeqLM`, on the character-level Penn Treebank with the recipe of
that section (`d2l.fit_lm`). Each head's state holds $32 \times 32$
numbers, so a layer holds 4,096, against 2,048 for the S4D layer of
that section with 16 states per channel.

In [ ]:
data = d2l.PennTreebank(batch_size=64, num_steps=128)

In [ ]:
rngs = nnx.Rngs(random.getrandbits(32))
model = d2l.SeqLM(len(data.vocab), 128, [GatedLinearAttention(
    128, num_heads=4, rngs=rngs) for _ in range(2)], rngs=rngs)
print(f'gated linear attention perplexity: {d2l.fit_lm(model, data):.2f}')

The model reaches a perplexity of about 3.1, better than the minGRU (3.2)
and S4D (3.35), behind the LSTM, the GRU, and Mamba (about 3.0).
How much of this comes from the parts around the memory? We remove the
short convolution, the SiLU, and the output gate, and keep the unit
queries and keys, the decays, and the output norm.

In [ ]:
class PlainLinearAttention(GatedLinearAttention):
    """Gated linear attention without the short convolution, the SiLU, and
    the output gate."""
    def __call__(self, X):
        Q, K, V = (heads_to_batch(Z, self.num_heads)
                   for Z in jnp.split(self.W_qkv(X), 3, -1))
        Q, K = (Z / jnp.maximum(jnp.linalg.norm(Z, axis=-1, keepdims=True),
                                1e-12) for Z in (Q, K))
        Y = self.memory(X, Q, K, V, self.decays(X))
        Y = Y / jnp.sqrt((Y ** 2).mean(-1, keepdims=True) + 1e-6)
        return self.W_o(batch_to_heads(Y, X.shape[0]))

In [ ]:
rngs = nnx.Rngs(random.getrandbits(32))
model = d2l.SeqLM(len(data.vocab), 128, [PlainLinearAttention(
    128, num_heads=4, rngs=rngs) for _ in range(2)], rngs=rngs)
print(f'without convolution, SiLU, and gate: {d2l.fit_lm(model, data):.2f}')

Without them, the same layer reaches only about 3.2. The
matrix memory itself does not help much on this task, as the models of
that section suggested: predicting the next character rarely requires
looking up a pair stored a hundred characters earlier. The tasks of
that section do, and there the size of the memory limits how much
it can recall.

One limitation is built into every model in this section. A write adds
$\mathbf{v}_t \mathbf{k}_t^\top$ to the memory, whatever the memory already
holds under $\mathbf{k}_t$. If a program sets `rate = 0.05` and later
`rate = 0.07`, the memory returns the sum of both values, scaled by the
decays, when queried with the key of `rate`. A decay can make the old
value fade, but it fades every other pair as well. The next section changes
the write so that it replaces the value stored under a key.

## Summary

A matrix-valued state stores key–value pairs: a write adds the outer
product $\mathbf{v}_t\mathbf{k}_t^\top$, and a read multiplies the state with
a query. Unrolled, this recurrence is causal linear attention, and the
selective state space model of that section is a memory of this
kind with the step-scaled input as value and $\mathbf{B}_t$ and
$\mathbf{C}_t$ as key and query. Without decay, every stored pair
interferes with every read: for random unit keys the expected squared read
error is $(n-1)/d_k$ after $n$ writes. An input-dependent decay lets old
pairs fade. With a scalar decay per head, the outputs equal masked attention
with a learned, decaying mask, $(\mathbf{L} \odot \mathbf{Q}\mathbf{K}^\top)
\mathbf{V}$, and computing this product in chunks, attention within a chunk
and a recurrence over the chunk boundaries, trains the model with matrix
products in memory linear in the length. A decay per key coordinate, as
in gated linear attention, gives each coordinate a mask of its own;
computing every decay over a stretch of time keeps all factors at most
one, while folding the decays into queries and keys overflows. The layer
surrounds the memory with a short convolution, unit queries and keys, and
a normalized, gated output. Every write adds, so a key that is written
twice returns the sum of its values.

## Exercises

1. **Mamba as a matrix memory.** Write the selective state space model of
   that section with its channel states as the rows of $\mathbf{S}_t
   \in \mathbb{R}^{H \times N}$, and drop the skip term.
    1. Write the update of $\mathbf{S}_t$ as an elementwise product of
       $\mathbf{S}_{t-1}$ with a decay matrix $\mathbf{E}_t$, plus an outer
       product, and the output as a matrix–vector product. Give every entry
       of $\mathbf{E}_t$.
    1. Under which condition on $\boldsymbol{\Delta}_t$ and on the rates
       $\lambda_n$ is $\mathbf{E}_t$ a scalar multiple of the matrix of
       ones, so that the update has the form the equation? Which of
       these conditions does Mamba-2 impose?
    1. For each form of decay in this section, count the input-dependent
       numbers per token that determine the decay of one head.

1. **Rank-one blocks.** Let $p_t = a_1 a_2 \cdots a_t$ be the products of
   the decays in the equation.
    1. Express $L_{ts}$ through the $p_t$, and show that every submatrix of
       $\mathbf{L}$ whose row indices are all at least as large as its
       column indices has rank one.
    1. Deduce that every block of $\mathbf{L} \odot
       \mathbf{Q}\mathbf{K}^\top$ that couples a chunk to an earlier chunk
       has rank at most $d_k$. Identify where `chunked_linear_attention`
       applies the query-side and the key-side decays of such a block.
    1. With a decay per key coordinate, $\mathbf{S}_t = \mathbf{S}_{t-1}
       \textrm{diag}(\boldsymbol{\alpha}_t) + \mathbf{v}_t
       \mathbf{k}_t^\top$, no single mask $\mathbf{L}$ multiplies
       $\mathbf{Q}\mathbf{K}^\top$. Write the decayed score of query $t$
       and key $s$, and show that a block coupling two chunks still has
       rank at most $d_k$.

1. **Counting operations.** Count one multiply-add as two floating-point
   operations, set $d_k = d_v = d$, and keep the leading terms.
    1. Count the operations of the recurrence, the dual form, and the
       chunked form at length $T$ and chunk size $m$. Split the chunked
       count into the three parts of the algorithm.
    1. Show that the chunked form costs at most a small constant times the
       recurrence while $m \le d$, and compute the ratio of chunked to dual
       operations at $T = 4096$, $d = 64$, and $m = 256$.
    1. Model the run time of a schedule as its operations divided by a
       throughput $P$, plus a fixed time $\tau$ per sequential step: $T$
       steps for the recurrence and $T/m$ for the chunked form. Find the
       chunk size that minimizes the chunked time, and evaluate it for
       $T = 4096$, $d = 64$, $P = 10^{13}$ operations per second, and
       $\tau = 5\,\mu\textrm{s}$. Compare with the timings above.

1. [code] **The normalizer as a value column.** The linear attention of
   that section applies a positive feature map $\phi$
   to queries and keys and divides each output by
   $\phi(\mathbf{q}_t)^\top \mathbf{z}_t$, where $\mathbf{z}_t =
   \sum_{s \le t} \phi(\mathbf{k}_s)$. Give $\mathbf{z}_t$ the same decay
   as the state.
    1. Show that appending a constant 1 to every value vector makes the
       last coordinate of the output equal to the denominator.
    1. Compute the normalized outputs with one call of
       `chunked_linear_attention`, with $\phi = \textrm{elu} + 1$, and check
       them against a loop that carries $\mathbf{S}_t$ and $\mathbf{z}_t$.
    1. Which of the two changes, the feature map or the normalizer, alters
       the relative weights that a query gives to past positions, and which
       only rescales each output?

1. **Targeted forgetting.** Let a memory hold $\mathbf{S} =
   \mathbf{v}\mathbf{k}^\top + \sum_i \mathbf{v}_i \mathbf{k}_i^\top$,
   where $\mathbf{k}$ has unit length and every $\mathbf{k}_i$ is
   orthogonal to $\mathbf{k}$.
    1. Find a matrix $\mathbf{D}$ such that $\mathbf{S}\mathbf{D} =
       \sum_i \mathbf{v}_i \mathbf{k}_i^\top$ for all such memories. Is it
       unique?
    1. For which keys $\mathbf{k}$ is this transition one of the decays of
       this section?
    1. Apply the transition and then write a new value,
       $\mathbf{S}\mathbf{D} + \mathbf{v}'\mathbf{k}^\top$. Show that the
       memory now returns $\mathbf{v}'$ for $\mathbf{k}$ and is otherwise
       unchanged. The next section builds a layer on this update.

1. **Correlated keys.** ● The keys with $\rho = 0.5$ in
   `matrix_memory` are normalized versions of $\sqrt{1-\rho^2}\,
   \mathbf{g}_i + \rho\, \mathbf{c}$, with independent random unit vectors
   $\mathbf{g}_i$ and one shared unit vector $\mathbf{c}$. Model the overlap
   of two distinct keys as $\mathbf{k}_i^\top \mathbf{k}_j = \rho^2 +
   \epsilon_{ij}$, where the $\epsilon_{ij}$ with $i \neq j$ are independent
   with mean zero and variance $(1-\rho^2)^2/d_k$, and neglect the
   normalization.
    1. Assume that the values are independent unit vectors with mean zero.
       Derive the expected squared read error at key $j$, which generalizes
       the equation, and identify the term that a larger
       $d_k$ does not reduce.
    1. The values of `matrix_memory` come from a codebook of 256 random
       vectors, whose mean $\bar{\mathbf{v}}$ is not exactly zero. Show
       that this adds a term proportional to $(n-1)(n-2)\,
       \|\bar{\mathbf{v}}\|^2$, and compare your complete prediction with
       the measured errors.

1. [code] **Decay as a sliding window.** ● Add a constant decay $a$ to
   `matrix_memory`, so that the pair written $k$ steps before the end
   enters the final memory with weight $a^k$.
    1. At $d_k = 64$ and $n = 256$, measure recall separately for the
       oldest and the newest quarter of the pairs, for $a \in \{1, 0.99,
       0.95, 0.9\}$. What does the decay give the newest pairs, and what
       does it cost the oldest?
    1. Following the proof of the proposition, show that the expected
       squared interference at any key is at most $1 / ((1-a^2)\, d_k)$.
       Show that the number of pairs whose squared signal $a^{2k}$ exceeds
       this bound is at most about $d_k / e$, whatever $a < 1$ is.

1. [code] **Sub-chunks.** ● The masks of `chunked_linear_attention` with
   a decay per key coordinate take $d_k m^2$ numbers per chunk.
    1. Compare their memory with that of a single mask, at $m = 64$ and
       $d_k = 64$.
    1. Cut each chunk into sub-chunks of 16 steps. For a query $t$ and a
       key $s$ in an earlier sub-chunk, let $r$ be the last step before
       the sub-chunk of $t$, and write $\exp(c_t - c_s) = \exp(c_t - c_r)
       \exp(c_r - c_s)$. Show that both factors lie in $(0, 1]$, so that
       the scores between sub-chunks are a plain matrix product of scaled
       queries and keys that cannot overflow.
    1. Implement the masks within sub-chunks and the products between them,
       check the result against the loop for chunk sizes 64 and 256, and
       compare memory and run time with `chunked_linear_attention`. The
       kernels of @Yang.Wang.Shen.ea.2024 work this way.

1. [code] **One part at a time.** `PlainLinearAttention` drops three
   parts of gated linear attention at once. Remove the short convolution,
   the SiLU, and the output gate one at a time, and train each variant with
   `d2l.fit_lm`. Which part accounts for most of the gap in perplexity?

[Discussions](https://d2l.discourse.group/)

[Discussions](https://d2l.discourse.group/t/36783)